# 1. 왜 게임 품질을 먼저 확인할까?

사람의 승패·실력이 아니라 정상 참여한 사람의 간격·리듬·행동·hold를 배우려는 실험입니다.
고의 포기 의심·비정상 장시간·AI만 계속된 흔적을 분리합니다. 이번에는 **검토 목록**만 만들고 학습 dataset이나 모델은 만들지 않습니다.
원본을 바꾸지 않고, include_candidate / exclude_candidate / review의 이유를 gameId와 source line으로 추적합니다.


In [ ]:
import json
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path

EXPERIMENT_ID = "human-game-quality-audit"
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
DEPENDENCY = ROOT / "scripts/ml/notebooks/human_action_data_readiness.ipynb"
shared = {"__name__": "human_game_quality_shared"}
for cell in json.loads(DEPENDENCY.read_text(encoding="utf-8"))["cells"]:
    if cell["cell_type"] == "code" and "definitions" in cell["metadata"].get("tags", []):
        exec(compile("".join(cell["source"]), str(DEPENDENCY), "exec"), shared)
# IO helper의 허용 report 폴더만 이번 실험으로 지정한다. 원본 Notebook 실행 셀은 실행하지 않는다.
shared["EXPERIMENT_ID"] = EXPERIMENT_ID
require, finite, integer = (shared[key] for key in ("require", "finite", "integer"))
iter_rows, normalize_event, scan_source = (shared[key] for key in ("iter_rows", "normalize_event", "scan_source"))
inspect_segment, interval_summary = (shared[key] for key in ("inspect_segment", "interval_summary"))
sha256_file, checked_paths, publish_report = (shared[key] for key in ("sha256_file", "checked_paths", "publish_report"))


# 2. 후보 기준은 채택된 정제 정책이 아닙니다

아래 30초·600초·5조각·60초는 **provisional-review-v1**의 검토용 기준입니다. 기존 V4의 기준을 승계하거나 학습 포함을 승인한 값이 아닙니다.
짧기만 하면 제외하지 않습니다. 30초 미만 + 종료 직전 5개 이상 연속 조각에서 hard_drop만 기록된 경우를 검토합니다.
600초 초과는 제외 후보, AI 기록만 남은 관찰 tail이 60초를 넘으면 검토입니다. 원인·포기 의도는 확정하지 않습니다.
Full 결과의 분포와 사례를 보고 사용자가 정책을 고정한 뒤 별도 dataset 단계로 진행합니다.


In [ ]:
DRAFT_POLICY = {"shortGameReviewSeconds": 30, "maxGameDurationSeconds": 600,
                "abandonmentTailMinPieces": 5, "aiOnlyTailReviewSeconds": 60}

def validate_policy(policy):
    require(isinstance(policy, dict) and set(policy) == set(DRAFT_POLICY), "INVALID_POLICY_KEYS")
    require(all(finite(value) and value > 0 for value in policy.values()), "INVALID_POLICY_NUMBER")
    require(integer(policy["abandonmentTailMinPieces"]) and policy["abandonmentTailMinPieces"] >= 2,
            "INVALID_ABANDONMENT_TAIL")
    require(policy["shortGameReviewSeconds"] < policy["maxGameDurationSeconds"], "INVALID_DURATION_BOUNDS")
    return dict(policy)

POLICY = validate_policy(DRAFT_POLICY)


# 3. 어떤 시간을 계산할까?

게임 길이는 game_start→game_over의 이벤트 시각입니다. DB row는 occurred_at, app event는 startedAt/endedAt을 사용합니다.
ingested_at으로 대체하지 않습니다. 개별 입력 createdAt은 runtime 처리 시도 시각이며 반응시간이 아닙니다.
Human과 AI의 시계는 다를 수 있습니다. AI-only tail은 마지막 Human **원본 행 뒤**의 AI 이벤트끼리의 시각 범위로만 진단하며, 실제 이탈 시간이나 버그 증거로 확정하지 않습니다.


In [ ]:
def event_time_ms(raw, event, payload):
    if "event_type" in raw:
        value = raw.get("occurred_at")
        if not isinstance(value, str):
            return None
        try:
            date = datetime.fromisoformat(value.replace("Z", "+00:00"))
            if date.tzinfo is None:
                return None
            delta = date.astimezone(timezone.utc) - datetime(1970, 1, 1, tzinfo=timezone.utc)
            result = delta.days * 86400000 + delta.seconds * 1000 + delta.microseconds / 1000
        except (ValueError, OverflowError):
            return None
    else:
        field = {"game_start": "startedAt", "game_over": "endedAt", "placement_decision": "createdAt"}[event["eventType"]]
        result = payload.get(field)
    return result if finite(result) and result >= 0 else None

def ending_hard_drop_tail(rows):
    ordered = sorted(rows, key=lambda row: (row["pieceIndex"], row["line"]))
    tail = []
    for row in reversed(ordered):
        if not row["hardDropOnly"] or (tail and row["pieceIndex"] != tail[-1]["pieceIndex"] - 1):
            break
        tail.append(row)
    return {"pieces": len(tail), "firstLine": tail[-1]["line"] if tail else None,
            "lastLine": tail[0]["line"] if tail else None}


# 4. 원본을 복제하지 않고 필요한 사실만 모읍니다

기존 Notebook의 streaming parser·identity 충돌 격리·구간 검사를 재사용합니다. 공통 index에 모든 유효 이벤트의 작은 metadata만 노출했습니다.
보드·원본 payload·전체 입력은 report에 복사하지 않습니다. playerId는 게임 안의 번호로 바꾸고 절대 시각도 내보내지 않습니다.
PVE의 AI는 진단용이고 학습 주체가 아닙니다. PVP의 양쪽 Human을 모두 봅니다.


In [ ]:
def collect_quality_facts(source, index, selected):
    games = {game: {"starts": [], "ends": [], "players": {}, "ai": [], "times": [],
                    "modes": set(), "difficulties": set(), "otherSources": 0} for game in selected}
    for line, raw, kind in iter_rows(source):
        meta = index["allValidByLine"].get(line)
        if meta is None or meta["gameId"] not in games:
            continue
        require(kind == "parsed", "SOURCE_CHANGED_DURING_AUDIT")
        _, event, payload, _ = normalize_event(raw)
        game = games[meta["gameId"]]
        stamp = event_time_ms(raw, event, payload)
        if stamp is not None:
            game["times"].append(stamp)
        mode = payload.get("mode")
        if mode is not None:
            game["modes"].add("pvp" if mode in ("multi", "multiplayer", "pvp") else "pve" if mode == "pve" else "other")
        difficulty = payload.get("aiDifficulty")
        if difficulty is not None:
            game["difficulties"].add(difficulty if difficulty in ("normal", "hard") else "invalid")
        if event["eventType"] == "game_start":
            game["starts"].append({"time": stamp, "line": line})
        elif event["eventType"] == "game_over":
            game["ends"].append({"time": stamp, "line": line, "finishType": payload.get("finishType"),
                                 "winner": payload.get("winnerPlayerId")})
        elif meta["source"] == "heuristic_ai":
            game["ai"].append({"time": stamp, "line": line})
        elif meta["source"] == "human":
            player = game["players"].setdefault(meta["playerId"], {"pieces": [], "actions": Counter()})
            inspected = inspect_segment(payload, meta, index["problems"].get(line, ()))
            inputs = payload.get("inputSequence")
            only_drop = (isinstance(inputs, list) and bool(inputs)
                         and inspected["readiness"]["logged_inter_action_timing"] in ("usable", "limited")
                         and all(isinstance(item, dict) and item.get("action") == "hard_drop" for item in inputs))
            player["pieces"].append({"pieceIndex": meta["pieceIndex"], "line": line,
                                     "hardDropOnly": only_drop,
                                     "timingUsable": inspected["readiness"]["logged_inter_action_timing"] == "usable",
                                     "holdAttempts": inspected["counts"].get("holdAttempts", 0)})
            player["actions"].update(inspected["actions"])
        else:
            game["otherSources"] += 1
    return games


# 5. 왜 분류됐는지 근거를 봅니다

목록은 include_candidate(구간 검사로 넘길 후보), exclude_candidate(현재 후보 기준에서 제외), review(근거 부족/의심)입니다.
세 목록 모두 **학습 승인 목록이 아닙니다**. 모호한 lifecycle·혼합 규칙·source 오류를 조용히 정상으로 취급하지 않습니다.
짧지만 이동·회전하다 진 정상 경기는 후보로 남을 수 있습니다. 한 Human의 포기 의심도 첫 보수적 정책에서는 게임 전체를 검토 대상으로 보냅니다.


In [ ]:
def classify_game(game_id, facts, index, policy, source_valid=True):
    reasons, excluded = set(), set()
    life = index["games"][game_id]
    start = facts["starts"][0]["time"] if len(facts["starts"]) == 1 else None
    end_row = facts["ends"][0] if len(facts["ends"]) == 1 else None
    end = end_row["time"] if end_row else None
    duration = (end - start) / 1000 if start is not None and end is not None and end > start else None
    if duration is None:
        reasons.add("GAME_DURATION_UNVERIFIED")
    if not life["completeNormal"]:
        reasons.add("LIFECYCLE_OR_FINISH_UNVERIFIED")
    if life["mixedRuleset"]:
        reasons.add("MIXED_RULESET")
    if not source_valid:
        reasons.add("SOURCE_CONTRACT_INVALID")
    mode = next(iter(facts["modes"])) if len(facts["modes"]) == 1 else "unspecified"
    if mode not in ("pve", "pvp"):
        reasons.add("MODE_UNVERIFIED")
    if len(facts["difficulties"]) > 1 or "invalid" in facts["difficulties"]:
        reasons.add("DIFFICULTY_AMBIGUOUS")
    difficulty = next(iter(facts["difficulties"])) if len(facts["difficulties"]) == 1 else "unspecified"
    if facts["otherSources"]:
        reasons.add("UNSUPPORTED_SOURCE")
    if duration is not None and duration > policy["maxGameDurationSeconds"]:
        excluded.add("ABNORMAL_LONG_GAME")
    people = []
    last_human_line = 0
    for person, player in sorted(facts["players"].items()):
        pieces = player["pieces"]
        tail = ending_hard_drop_tail(pieces)
        if len({row["pieceIndex"] for row in pieces}) != len(pieces):
            reasons.add("HUMAN_LINEAGE_UNVERIFIED")
        if (duration is not None and duration < policy["shortGameReviewSeconds"]
                and tail["pieces"] >= policy["abandonmentTailMinPieces"]):
            reasons.add("EARLY_ABANDONMENT_SUSPECTED")
        last_human_line = max(last_human_line, max(row["line"] for row in pieces))
        winner = end_row["winner"] if end_row else None
        outcome = "unknown" if not isinstance(winner, str) or not winner else "win" if winner == person else "loss"
        people.append({"playerNumber": index["pseudonyms"][game_id][person], "outcome": outcome,
                       "humanSegments": len(pieces), "actionCounts": dict(player["actions"]),
                       "endingHardDropTail": tail,
                       "timingNoHoldSegments": sum(row["timingUsable"] and not row["holdAttempts"] for row in pieces)})
    if not people:
        excluded.add("NO_HUMAN_DATA")
    trailing_ai = [row for row in facts["ai"] if row["line"] > last_human_line and row["time"] is not None]
    tail_span = ((max(row["time"] for row in trailing_ai) - min(row["time"] for row in trailing_ai)) / 1000
                 if len(trailing_ai) >= 2 else None)
    if people and tail_span is not None and tail_span > policy["aiOnlyTailReviewSeconds"]:
        reasons.add("AI_ONLY_CONTINUATION_SUSPECTED")
    status = "exclude_candidate" if excluded else "review" if reasons else "include_candidate"
    return {"gameId": game_id, "status": status, "reasons": sorted(reasons | excluded),
            "mode": mode, "difficulty": difficulty, "durationSeconds": duration,
            "observedSpanSeconds": ((max(facts["times"]) - min(facts["times"])) / 1000 if facts["times"] else None),
            "observedSpanClock": "mixed_event_clocks_unverified",
            "lifecycleLines": {"start": [row["line"] for row in facts["starts"]], "end": [row["line"] for row in facts["ends"]]},
            "aiPlacementCount": len(facts["ai"]), "trailingAiCount": len(trailing_ai),
            "aiTailObservedSpanSeconds": tail_span, "aiTailBasis": "source_order_then_ai_clock_span_diagnostic_only",
            "humanPlayers": people, "segmentQualityRequired": True}


# 6. 손으로 따라 보는 작은 예제

아래는 실데이터가 아닙니다. 15초라도 이동·회전 기록이 있으면 길이만으로 제외하지 않고, 5개 조각 연속 hard_drop만 남았다면 검토합니다.
601초면 현재 후보 기준의 제외 대상입니다. lifecycle 누락은 duration을 추정하지 않고 review로 둡니다.


In [ ]:
def worked_example(seconds=15, hard_only=False):
    pieces = [{"pieceIndex": i, "line": i + 2, "hardDropOnly": hard_only,
               "timingUsable": True, "holdAttempts": 0} for i in range(5)]
    facts = {"starts": [{"time": 0, "line": 1}], "ends": [{"time": seconds * 1000, "line": 8,
             "finishType": "normal", "winner": "synthetic-person"}],
             "players": {"synthetic-person": {"pieces": pieces, "actions": Counter({"hard_drop": 5} if hard_only else {"move_left": 5, "hard_drop": 5})}},
             "ai": [], "times": [0, seconds * 1000], "modes": {"pve"}, "difficulties": {"hard"}, "otherSources": 0}
    index = {"games": {"synthetic-game": {"completeNormal": True, "mixedRuleset": False}},
             "pseudonyms": {"synthetic-game": {"synthetic-person": 1}}}
    return classify_game("synthetic-game", facts, index, POLICY)


In [ ]:
print("Synthetic only:", {name: value["status"] for name, value in {
    "short_normal": worked_example(), "short_drop_tail": worked_example(hard_only=True),
    "too_long": worked_example(seconds=601)}.items()})


# 7. Source 전체 metadata와 smoke 상세 분석을 구분합니다

원본 전체를 streaming scan해 identity를 확인합니다. smoke는 Human 첫 3게임과 가능하면 Human 없는 첫 게임 1개만 자세히 봅니다.
full은 모든 game을 봅니다. 게임 분류 합계와 Human 구간 합계는 재검증합니다. 중복은 한 번, 충돌은 원본 parser처럼 격리합니다.
실행 전후 SHA가 바뀌면 report를 게시하지 않습니다. 기존 출력도 덮어쓰지 않습니다.


In [ ]:
def audit_game_quality(source, source_label, mode="smoke", policy=None):
    require(mode in ("smoke", "full"), "INVALID_RUN_MODE")
    policy = validate_policy(DRAFT_POLICY if policy is None else policy)
    before = sha256_file(source)
    dependency_sha = sha256_file(DEPENDENCY)
    index = scan_source(source)
    all_games = sorted(index["games"])
    require(all_games, "NO_GAMES")
    if mode == "full":
        selected = all_games
    else:
        selected = index["humanGameIds"][:3]
        extra = next((g for g in all_games if g not in set(index["humanGameIds"])), None)
        if extra is not None:
            selected.append(extra)
        selected = sorted(selected)
    require(selected, "NO_SMOKE_COHORT")
    counts = index["counts"]
    source_valid = not any(counts.get(k, 0) for k in ("parse_failure", "invalid", "identityConflictRows"))
    facts = collect_quality_facts(source, index, selected)
    rows = [classify_game(g, facts[g], index, policy, source_valid) for g in selected]
    statuses = {name: sum(r["status"] == name for r in rows)
                for name in ("include_candidate", "exclude_candidate", "review")}
    require(sum(statuses.values()) == len(selected), "GAME_CLASSIFICATION_COUNT_MISMATCH")
    expected_human = sum(m["gameId"] in set(selected) for m in index["validByLine"].values())
    actual_human = sum(p["humanSegments"] for r in rows for p in r["humanPlayers"])
    require(expected_human == actual_human, "HUMAN_COUNT_MISMATCH")
    if mode == "full":
        require(actual_human == index["humanCount"], "FULL_HUMAN_COUNT_MISMATCH")
    strata = defaultdict(Counter)
    outcomes = defaultdict(Counter)
    reasons = Counter()
    lists = {status: [] for status in statuses}
    for row in rows:
        strata[(row["mode"], row["difficulty"])][row["status"]] += 1
        lists[row["status"]].append(row["gameId"])
        reasons.update(row["reasons"])
        for person in row["humanPlayers"]:
            outcomes[person["outcome"]][row["status"]] += 1
    require(before == sha256_file(source), "SOURCE_CHECKSUM_CHANGED")
    require(dependency_sha == sha256_file(DEPENDENCY), "DEPENDENCY_CHECKSUM_CHANGED")
    return {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID,
            "status": ("human_game_quality_smoke_valid" if mode == "smoke" else "needs_quality_policy_review")
                      if source_valid else "source_contract_invalid",
            "runMode": mode, "fullCohortCovered": mode == "full", "qualityReviewComplete": mode == "full",
            "dataDecisionEligible": False, "policyDecisionEligible": False, "canonicalResultEligible": False,
            "modelFitCount": 0, "datasetExported": False, "policyStatus": "provisional_review_only",
            "policy": policy, "sourceUnchanged": True, "sourceContractValid": source_valid,
            "source": {"path": source_label, "sha256": before, "bytes": Path(source).stat().st_size},
            "dependencyNotebookSha256": dependency_sha,
            "sourceScan": {"counts": counts, "placementBySource": index["placementBySource"], "games": len(all_games)},
            "cohort": {"selection": "all_games" if mode == "full" else "first_three_human_plus_first_without_human",
                       "gameIds": selected, "fullSourceGameCount": len(all_games)},
            "summary": {"classificationCounts": statuses, "reasonCounts": dict(reasons),
                        "humanSegments": actual_human,
                        "durationSeconds": interval_summary([r["durationSeconds"] for r in rows if r["durationSeconds"] is not None]),
                        "byModeDifficulty": [{"mode": m, "difficulty": d, "counts": dict(c)} for (m, d), c in sorted(strata.items())],
                        "byHumanOutcome": {key: dict(value) for key, value in sorted(outcomes.items())}},
            "lists": lists, "games": rows,
            "limits": ["candidate_policy_not_adopted", "source_order_is_not_execution_order",
                       "clock_origins_not_bound_to_historical_build", "no_proven_abandonment_or_bug",
                       "no_dataset_or_model", "hold_context_and_replay_still_unverified"]}

def run_game_quality_from_environment():
    import os
    mode = os.environ.get("HUMAN_GAME_QUALITY_RUN_MODE", "smoke")
    require(mode != "full" or "HUMAN_GAME_QUALITY_REPORT" in os.environ, "FULL_REPORT_PATH_REQUIRED")
    policy = json.loads(os.environ["HUMAN_GAME_QUALITY_POLICY"], parse_constant=shared["reject_constant"]) if "HUMAN_GAME_QUALITY_POLICY" in os.environ else DRAFT_POLICY
    source, output = checked_paths(ROOT, os.environ.get("HUMAN_GAME_QUALITY_SOURCE", shared["DEFAULT_SOURCE"]),
                                   os.environ.get("HUMAN_GAME_QUALITY_REPORT", "/private/tmp/human-game-quality-smoke.json"), mode)
    report = audit_game_quality(source, source.relative_to(ROOT).as_posix(), mode, policy)
    publish_report(report, output)
    return report


# 8. 실행과 결과 확인

처음에는 smoke입니다. full은 명세의 HUMAN_GAME_QUALITY_RUN_MODE=full과 새 report 경로를 명시해 실행합니다.
출력 이름에 full을 쓰는 것만으로 full이 되지 않습니다. Error가 나면 다음 단계로 넘어가지 않습니다.


In [ ]:
report = run_game_quality_from_environment()
print(json.dumps({key: report[key] for key in ("status", "runMode", "policyStatus", "summary")}, ensure_ascii=False, indent=2))


# 9. pandas 표와 Matplotlib 그래프로 분모·편향·근거를 읽습니다

분류·난이도·사유 표는 **게임 단위**입니다. 사유는 중복될 수 있어 합계가 게임 수와 다를 수 있습니다.
승패 표는 **Human player 단위**여서 PVP 한 경기가 win/loss 양쪽에 나타날 수 있습니다.
길이 histogram은 시작·종료가 확인된 경기만 포함하며 누락을 0초로 채우지 않습니다. AI tail·observedSpanSeconds는 실제 게임 길이가 아닙니다.
include_candidate는 학습 승인, review는 포기·버그 확정이 아닙니다. 그래프 제목에 smoke/full을 표시합니다.
검토 목록의 gameId·reasons를 보고 report의 lifecycle source line·마지막 hard_drop 근거로 돌아갑니다.
기존 full 결과만 다시 보려면 감사 실행 셀 대신 아래를 실행한 뒤 시각화 셀을 실행합니다.

```python
report = json.loads((ROOT / "data/ml-experiments/human-game-quality-audit/full-report.json").read_text(encoding="utf-8"))
```


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

classification_order = ["include_candidate", "exclude_candidate", "review"]
game_table = pd.DataFrame([{
    **{key: row[key] for key in ("gameId", "mode", "difficulty", "durationSeconds", "status", "reasons")},
    "humanSegments": sum(player["humanSegments"] for player in row["humanPlayers"]),
    "maxHardDropTail": max((player["endingHardDropTail"]["pieces"] for player in row["humanPlayers"]), default=0),
    "aiTailObservedSpanSeconds": row["aiTailObservedSpanSeconds"],
} for row in report["games"]], columns=["gameId", "mode", "difficulty", "durationSeconds", "status", "reasons", "humanSegments", "maxHardDropTail", "aiTailObservedSpanSeconds"])
classification_table = game_table["status"].value_counts().reindex(classification_order, fill_value=0).to_frame("games")
classification_table["percent"] = (classification_table["games"] / max(len(game_table), 1) * 100).round(2)
mode_table = pd.crosstab([game_table["mode"], game_table["difficulty"]], game_table["status"]).reindex(columns=classification_order, fill_value=0)
outcome_table = pd.DataFrame(report["summary"]["byHumanOutcome"]).T.reindex(columns=classification_order, fill_value=0).fillna(0).astype(int)
reason_table = pd.Series(report["summary"]["reasonCounts"], dtype="int64", name="games").sort_values(ascending=False).to_frame()
review_table = game_table.loc[game_table["status"] == "review"].copy()
durations = pd.to_numeric(game_table["durationSeconds"], errors="raise").dropna()

print(f"Mode: {report['runMode']} | Games: {len(game_table)} | Human segments: {report['summary']['humanSegments']:,} | Policy: {report['policyStatus']}")
print("Classification (games; candidate lists only)")
display(classification_table)
print("Mode / difficulty (games)")
display(mode_table)
print("Outcome (Human players, not games; winners are not a selection rule)")
display(outcome_table)
print("Reasons (overlap allowed)")
display(reason_table)
print(f"Verified lifecycle duration: {len(durations)} games; missing: {len(game_table) - len(durations)}")
display(pd.Series(report["summary"]["durationSeconds"], name="seconds_except_count").to_frame())
print("Review cases (not confirmed abandonment or bugs)")
display(review_table)

fig, axes = plt.subplots(2, 2, figsize=(14, 9), layout="constrained")
classification_table["games"].plot.bar(ax=axes[0, 0], color=["#0072B2", "#D55E00", "#E69F00"], rot=15)
axes[0, 0].set(title="Candidate classification", xlabel="Classification", ylabel="Games")
if mode_table.empty:
    axes[0, 1].text(0.5, 0.5, "No mode / difficulty rows", ha="center", transform=axes[0, 1].transAxes)
else:
    mode_table.plot.bar(stacked=True, ax=axes[0, 1], color=["#0072B2", "#D55E00", "#E69F00"], rot=15)
axes[0, 1].set(title="Mode / difficulty", xlabel="Mode, difficulty", ylabel="Games")
if durations.empty:
    axes[1, 0].text(0.5, 0.5, "No verified lifecycle duration", ha="center", transform=axes[1, 0].transAxes)
else:
    axes[1, 0].hist(durations, bins=min(20, len(durations)), color="#0072B2", edgecolor="white")
    axes[1, 0].axvline(report["policy"]["shortGameReviewSeconds"], color="#D55E00", linestyle="--", label="Short-game review boundary (not automatic exclusion)")
    axes[1, 0].legend(fontsize="small")
axes[1, 0].set(title=f"Verified lifecycle duration (n={len(durations)}, missing={len(game_table) - len(durations)})", xlabel="Seconds", ylabel="Games")
if reason_table.empty:
    axes[1, 1].text(0.5, 0.5, "No flagged reasons", ha="center", transform=axes[1, 1].transAxes)
else:
    reason_table.sort_values("games").plot.barh(ax=axes[1, 1], legend=False, color="#E69F00")
axes[1, 1].set(title="Review / exclusion reasons (overlap allowed)", xlabel="Games", ylabel="Reason")
fig.suptitle(f"Human game quality | {report['runMode']} | provisional review only")
plt.show()
plt.close(fig)


# 10. 한계와 다음 gate

이 결과는 학습용 포함 승인이나 사람다운 controller의 성능이 아닙니다. 정상 빠른 패배·긴 정상 플레이가 과잉 제외되는지 사례를 확인해야 합니다.
다음은 full 결과를 검토해 정책 수치를 고정하는 것입니다. 그 뒤 새 명세에서 Human 구간 정제·간격 dataset·game split을 만들며 지금은 자동 시작하지 않습니다.
Hold 정책은 성공·직전 상태·replay가 별도 검증될 때까지 보류합니다. 원본·이전 감사 report·모델·runtime은 변경하지 않습니다.
